# Fase 18b: Auditoría de datos temporales - frecuencias, leakage y multicolinealidad

## Motivación

El dataset contiene 60 variables con **frecuencias de actualización muy
dispares**: diarias (tipos, FX, materias primas), mensuales (CPI, paro, M2),
trimestrales (PIB). Esta disparidad plantea tres riesgos metodológicos que
esta auditoría cuantifica:

1. **Lookahead bias**: las variables macro se publican con retraso (el CPI
   de enero se conoce en febrero). Si el dataset las trae "adelantadas",
   el modelo aprende con información del futuro.
2. **Consistencia del forward-fill**: rellenar huecos con el último valor
   conocido es correcto SOLO si no se cruza el momento de publicación.
3. **Multicolinealidad**: variables económicas correlacionan entre sí
   (VIF alto), lo que puede inflar la varianza de los coeficientes.

---

> **Aviso de auditoria:** las cifras y salidas mostradas en este notebook corresponden a una ejecucion anterior a las correcciones metodologicas. Es necesario reejecutar el pipeline completo antes de usar o comunicar sus resultados.


Configuración del notebook (raíz del proyecto).

In [1]:
import sys
from pathlib import Path

def _find_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in (p, *p.parents):
        if (candidate / "configs" / "config.yaml").is_file():
            return candidate
    raise FileNotFoundError(
        "No se encontro configs/config.yaml desde el directorio de trabajo. "
        "Abra Jupyter dentro del repositorio."
    )

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.utils import path_from_root, set_publication_style, set_seed
set_seed(42)
set_publication_style()

Carga de datos crudos y limpios.

`raw` contiene las series originales (con huecos reales); `clean` es el
resultado del pipeline de limpieza (ffill, días hábiles, ventana 2000-2025).
Comparar ambos revela cómo se trataron los huecos.

In [2]:
import numpy as np
import pandas as pd

from src.config import get_config
from src.data.load_data import clean_daily_series, load_raw

cfg = get_config()
raw = load_raw(cfg)
clean = clean_daily_series(raw, cfg)
print(f"raw: {raw.shape} | clean: {clean.shape}")
print(f"Ventana: {clean['date'].min().date()} -> {clean['date'].max().date()}")

  [clean] filas con target: 6705 (descartadas 0)
raw: (45368, 61) | clean: (6705, 27)
Ventana: 2000-01-03 -> 2025-09-12


1. Frecuencia de actualización real por feature.

Medimos el intervalo mediano entre cambios de valor de cada serie en la
ventana 2000+. Esto clasifica cada variable por su frecuencia real:
diaria, semanal, mensual, trimestral, anual.

In [3]:
print("=== Frecuencia de actualización (días entre cambios) ===")
freqs = {}
for c in clean.columns:
    if c in ("date", "gold_spot"):
        continue
    s = clean[c]
    changes = s[s.diff() != 0].index
    if len(changes) > 1:
        freqs[c] = np.median(np.diff(changes))
    else:
        freqs[c] = np.nan

freq_s = pd.Series(freqs).sort_values()
print(freq_s.round(1).to_string())

def classify(d):
    if np.isnan(d):
        return "sin datos"
    if d <= 2:
        return "diaria"
    if d <= 8:
        return "semanal"
    if d <= 35:
        return "mensual"
    if d <= 100:
        return "trimestral"
    return "anual+"

types = pd.Series({c: classify(d) for c, d in freqs.items()})
print("\n=== Resumen por frecuencia ===")
print(types.value_counts().to_string())

=== Frecuencia de actualización (días entre cambios) ===
us10y_yield              1.0
policy_uncertainty       1.0
geopolitical_risk        1.0
usdjpy_exchange          1.0
silver_futures           1.0
eurusd_exchange          1.0
gold_futures             1.0
us2y_yield               1.0
usdinr_exchange          1.0
dxy_index                1.0
silver_spot              1.0
usdcny_exchange          1.0
wti_futures              1.0
palladium_spot           1.0
platinum_spot            1.0
dxy_future               1.0
wti_spot                 1.0
brent_spot               1.0
brent_futures            1.0
copper_futures           1.0
vix_index                1.0
commodities_bloomberg    1.0
commodities_crb          1.0
credit_spread            1.0
sp500_futures            1.0

=== Resumen por frecuencia ===
diaria    25


2. LOOKAHEAD BIAS: ¿cuándo se actualiza cada macro?

Comparamos el día del mes en que cambia cada variable macro contra su
calendario real de publicación:
- CPI (BLS): ~día 10-15 del mes siguiente.
- Paro (BLS): primer viernes del mes siguiente.
- PIB (BEA): ~30 días tras el cierre del trimestre.
- M2 (Fed): semanal con ~1 semana de retraso.

Si el dataset actualiza el día 1 del mes, el valor está ADELANTADO
(información que no existía en ese momento) -> fuga de información.

In [4]:
sub = raw[(raw["date"] >= "2000-01-01") & (raw["date"].dt.dayofweek < 5)].copy()
sub = sub.set_index("date")

print("=== Dia del mes de actualizacion (mediana) ===")
update_days = {}
for c in ["us_cpi", "us_unemployment", "us_gdp", "us_m2", "us_retail_sales",
          "us_industrial_production", "us_consumer_sentiment"]:
    if c not in sub.columns:
        continue
    s = sub[c].dropna()
    if len(s) < 3:
        continue
    change_dates = s.index[s.diff() != 0]
    days = pd.Series(change_dates.day)
    update_days[c] = {
        "median_day": float(days.median()),
        "n_changes": int(len(change_dates)),
    }
    print(f"{c:<28} dia={days.median():5.1f} (P10={days.quantile(0.1):4.0f}, "
          f"P90={days.quantile(0.9):4.0f}) | n_cambios={len(change_dates):4d}")

print("""
INTERPRETACIÓN:
- Si el día mediano es 1-5, el valor aparece al INICIO del mes -> el dataset
  no respeta el retraso de publicación real -> LOOKAHEAD BIAS.
- El forward-fill NO corrige esto: rellena hacia adelante, pero el primer
  valor ya está contaminado.
- Corrección profesional: aplicar publication lag (desplazar la serie k días
  hábiles) antes de construir features. El lag simula el momento en que el
  dato es realmente público.
""")

=== Día del mes de actualización (mediana) ===
us_cpi                       día=  1.0 (P10=   1, P90=   1) | n_cambios= 216
us_unemployment              día=  1.0 (P10=   1, P90=   1) | n_cambios= 166
us_gdp                       día=  1.0 (P10=   1, P90=   1) | n_cambios=  70
us_m2                        día=  1.0 (P10=   1, P90=   1) | n_cambios= 218
us_retail_sales              día=  1.0 (P10=   1, P90=   1) | n_cambios= 218
us_industrial_production     día=  1.0 (P10=   1, P90=   1) | n_cambios= 218
us_consumer_sentiment        día=  1.0 (P10=   1, P90=   1) | n_cambios= 216

INTERPRETACIÓN:
- Si el día mediano es 1-5, el valor aparece al INICIO del mes -> el dataset
  no respeta el retraso de publicación real -> LOOKAHEAD BIAS.
- El forward-fill NO corrige esto: rellena hacia adelante, pero el primer
  valor ya está contaminado.
- Corrección profesional: aplicar publication lag (desplazar la serie k días
  hábiles) antes de construir features. El lag simula el momento en que el
  

3. Publication lag: experimento no concluyente sin datos de fecha de publicacion.

Los lags propuestos no se pueden aplicar de forma valida sobre el parquet de
features ya construido: hace falta desplazar las series macro antes de
`build_features` usando fechas y horas reales de publicacion o datos vintage.
Por tanto esta fase no cuantifica impacto AUC y no interpreta una diferencia
entre quitar columnas como efecto de publication lag.

In [ ]:
PUBLICATION_LAG = {
    "us_cpi": 15, "us_unemployment": 5, "us_gdp": 30, "us_m2": 15,
    "fed_funds": 5, "us_industrial_production": 20, "us_retail_sales": 15,
    "us_consumer_sentiment": 15, "consumer_confidence": 25,
    "us10y_real": 15, "export_price_index": 15, "fx_reserves_china": 30,
    "us_personal_saving_rate": 30,
}

print("=== Publication lags propuestos (dias habiles) ===")
for k, v in PUBLICATION_LAG.items():
    print(f"  {k:<30} {v:>3}")

# Contexto comun que necesitan las celdas siguientes (VIF, cobertura, guardado).
import json

from src.data.split import drop_warmup, purge_label_overlap, temporal_split

feats = pd.read_parquet(path_from_root("data/processed/features.parquet"))
feats = drop_warmup(feats, warmup=260)
parts = temporal_split(feats, cfg)
# Purga del solape de etiquetas entre particiones consecutivas.
parts = purge_label_overlap(parts, cfg=cfg)
with open(path_from_root("models/feature_list.json"), encoding="utf-8") as f:
    sel_cols = json.load(f)

macro_feats = [c for c in sel_cols if any(m in c for m in PUBLICATION_LAG)]
non_macro = [c for c in sel_cols if c not in macro_feats]
print(f"\nFeatures macro potencialmente adelantadas: {len(macro_feats)}/{len(sel_cols)}")

# EXPERIMENTO NO CONCLUYENTE (hallazgo de auditoria):
# La version anterior comparaba el AUC con y sin las columnas macro y publicaba
# la diferencia como "impacto del publication lag". Eso mide la aportacion
# predictiva de esas columnas, NO el efecto de conocerlas antes de su
# publicacion: no se aplicaba ningun desplazamiento temporal. Para cuantificar
# el impacto real hacen falta las fechas de publicacion (vintages), que este
# dataset no conserva; el procedimiento correcto seria alinear cada serie por
# su fecha de publicacion, reconstruir features, purgar etiquetas y repetir el
# mismo protocolo temporal.
auc_full = None
auc_nom = None
print("""
EXPERIMENTO NO CONCLUYENTE:
- El dataset no conserva fechas de publicacion ni vintages.
- Quitar columnas macro no aplica lags ni mide su impacto causal sobre el AUC.
- impacto_auc se persiste como null en lugar de 0.0 para no sugerir ausencia de fuga.
""")


4. Multicolinealidad: VIF de las features seleccionadas.

El Variance Inflation Factor (VIF) mide cuánto infla la varianza de un
coeficiente la correlación con el resto. VIF > 10 indica multicolinealidad
severa. En series financieras es esperable (lags del mismo activo), pero
debe cuantificarse.

In [6]:
from sklearn.linear_model import LinearRegression

X = parts["train"][sel_cols].to_numpy(np.float64)
vifs = {}
for i, c in enumerate(sel_cols):
    y = X[:, i]
    X_rest = np.delete(X, i, axis=1)
    r2 = LinearRegression().fit(X_rest, y).score(X_rest, y)
    vifs[c] = 1 / (1 - r2) if r2 < 0.9999 else float("inf")

vif_s = pd.Series(vifs).sort_values(ascending=False)
print("=== Top 10 VIF ===")
print(vif_s.head(10).round(1).to_string())
print(f"\nFeatures con VIF > 10: {(vif_s > 10).sum()}/{len(sel_cols)}")
print(f"Features con VIF > 5: {(vif_s > 5).sum()}/{len(sel_cols)}")
print("""
INTERPRETACIÓN:
- VIF alto es esperable: gold_spot_lag1..lag21 correlacionan entre sí por
  construcción (lags de la misma serie).
- El modelo final (Ridge) es ROBUSTO a la multicolinealidad (regularización
  L2 estabiliza los coeficientes). Los árboles (RF) también la toleran.
- El filtro de correlación |rho|>0.98 ya eliminó los pares más extremos.
""")

=== Top 10 VIF ===
commodities_bloomberg    206.7
commodities_crb          177.8
gold_spot_lag21          176.9
year                     146.7
dxy_index                144.2
eurusd_exchange          107.5
sp500_futures             73.9
usdcny_exchange           39.5
usdinr_exchange           36.0
platinum_spot             35.4

Features con VIF > 10: 37/83
Features con VIF > 5: 51/83

INTERPRETACIÓN:
- VIF alto es esperable: gold_spot_lag1..lag21 correlacionan entre sí por
  construcción (lags de la misma serie).
- El modelo final (Ridge) es ROBUSTO a la multicolinealidad (regularización
  L2 estabiliza los coeficientes). Los árboles (RF) también la toleran.
- El filtro de correlación |rho|>0.98 ya eliminó los pares más extremos.



5. Rango de fechas óptimo y cobertura.

Verificamos que la ventana 2000-2025 tiene cobertura COMPLETA (100%) en
todas las features tras warm-up. Esto confirma que el rango elegido es
óptimo: antes de 2000 la cobertura cae drásticamente.

In [7]:
feats_cov = pd.read_parquet(path_from_root("data/processed/features.parquet"))
feats_cov = drop_warmup(feats_cov, warmup=260)
feats_cov["year"] = feats_cov["date"].dt.year
cov = feats_cov.groupby("year")[sel_cols].apply(
    lambda g: g.notna().mean().mean())
print(cov.round(3).to_string())
print(f"\nCobertura media 2001-2025: {cov.mean():.4f}")
print(f"Mínimo por año: {cov.min():.4f}")
print("""
CONCLUSIÓN: la ventana 2000-2025 es la adecuada. Todo el rango tiene
cobertura completa tras el warm-up (260 días) y el ffill. Antes de 2000,
la mayoría de series no existían (ver notebook 02).
""")

year
2001    1.0
2002    1.0
2003    1.0
2004    1.0
2005    1.0
2006    1.0
2007    1.0
2008    1.0
2009    1.0
2010    1.0
2011    1.0
2012    1.0
2013    1.0
2014    1.0
2015    1.0
2016    1.0
2017    1.0
2018    1.0
2019    1.0
2020    1.0
2021    1.0
2022    1.0
2023    1.0
2024    1.0
2025    1.0

Cobertura media 2001-2025: 1.0000
Mínimo por año: 1.0000

CONCLUSIÓN: la ventana 2000-2025 es la adecuada. Todo el rango tiene
cobertura completa tras el warm-up (260 días) y el ffill. Antes de 2000,
la mayoría de series no existían (ver notebook 02).



6. Guardado de resultados.

Persistimos la auditoría en reports/ para el informe técnico.

In [8]:
result = {
    "frecuencias": {c: (None if np.isnan(d) else float(d))
                    for c, d in freqs.items()},
    "lookahead": {
        "macro_update_days": update_days,
        "us_cpi_dia_actualizacion": update_days.get("us_cpi", {}).get("median_day"),
        "publication_lag_propuesto": PUBLICATION_LAG,
        "experimento_publication_lag": "no_concluyente_sin_vintages",
        "impacto_auc": None,
        "auc_con_macro": None,
        "auc_sin_macro": None,
        "nota": ("No se aplicaron lags reales porque faltan fechas de publicacion. "
                 "Quitar columnas macro no cuantifica el impacto de publication lag."),
    },
    "multicolinealidad": {
        "n_vif_gt_10": int((vif_s > 10).sum()),
        "n_vif_gt_5": int((vif_s > 5).sum()),
        "top_vif": {c: float(v) for c, v in vif_s.head(10).items()},
    },
    "cobertura": {str(y): float(v) for y, v in cov.items()},
    "conclusion": (f"Publication lag no cuantificado sin vintages. "
                   f"Multicolinealidad: {int((vif_s > 10).sum())}/{len(sel_cols)} features "
                   "con VIF > 10; tolerada por Ridge/RF. Ventana 2000-2025."),
}
with open(path_from_root("reports", "data_audit.json"), "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, default=float)
print("Guardado en reports/data_audit.json")

Guardado en reports/data_audit.json
